# PWN Leptonic Model Fitting: Inverse Compton (1 Population)

This notebook fits the leptonic (Inverse Compton) model to the combined Fermi-LAT + H.E.S.S. gamma-ray spectrum,
following the methodology from **Hnatyk et al. 2022 (MNRAS)**.

## Model Description
- **Physics**: Relativistic electrons scatter background photon fields (CMB, IR, starlight)
  to produce gamma-rays via Inverse Compton (IC) scattering
- **Electron spectrum**: Exponential Cutoff Power-Law (ECPL) - single population
- **Parameters**: Normalization, spectral index Γ, cutoff energy E_cut
- **Distance**: 12.5 kpc

## Seed Photon Fields (from Hnatyk et al. 2022)
- CMB: T = 2.7 K, w = 0.26 eV/cm³
- IR/FIR: T = 107 K, w = 1.19 eV/cm³  
- Starlight/NIR: T = 7906 K, w = 1.92 eV/cm³

## Reference
Hnatyk et al. 2022, MNRAS, Table 1 (Alternative ECPL model):
- Γ = 3.08 ± 0.03, E_cut = 424.3 ± 21.1 TeV, W_e = 8.80×10⁴⁹ erg

In [ ]:
import numpy as np
import astropy.units as u
from astropy.table import QTable
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

import naima
from naima.models import ExponentialCutoffPowerLaw, InverseCompton

# Import our custom modules
from spectrum_builder import build_spectrum_for_naima
from naima_models.pwn_inverse_compton import (
    pwn_ic_ecpl,
    pwn_lnprior_ecpl,
    PWN_DEFAULTS,
    SEED_PHOTON_FIELDS,
    compute_electron_energy,
    get_initial_params_ecpl,
    get_labels_ecpl,
)

## 1. Load or Build Spectrum Data

Option A: Use synthetic spectrum from published parameters  
Option B: Load actual Fermi-LAT data from file (when available)

In [ ]:
# Build spectrum (use fermi_file=None for synthetic, or path to actual data)
FERMI_DATA_FILE = None  # Set to path when you have actual Fermi data

data = build_spectrum_for_naima(
    fermi_file=FERMI_DATA_FILE,
    energy_min=0.2 * u.GeV,  # 200 MeV lower bound
    n_fermi_bins=10,
    n_hess_bins=10,
    use_hawc=False,
)

print(f"Spectrum has {len(data)} energy bins")
print(f"Energy range: {data['energy'].min():.2f} - {data['energy'].max():.2f}")
data

In [ ]:
# Plot the input spectrum
fig, ax = plt.subplots(figsize=(10, 6))

E = data['energy'].to(u.GeV)
flux = data['flux'].to(u.Unit('1/(cm2 s GeV)'))
flux_err = data['flux_error'].to(u.Unit('1/(cm2 s GeV)'))

# E² dN/dE (SED)
sed = (E**2 * flux).to(u.Unit('GeV/(cm2 s)'))
sed_err = (E**2 * flux_err).to(u.Unit('GeV/(cm2 s)'))

ax.errorbar(E.value, sed.value, yerr=sed_err.value, fmt='o', 
            color='blue', ecolor='blue', capsize=3, label='Fermi-LAT + H.E.S.S.')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Energy [GeV]')
ax.set_ylabel(r'$E^2 dN/dE$ [GeV cm$^{-2}$ s$^{-1}$]')
ax.set_title('Input Gamma-Ray Spectrum')
ax.legend()
ax.grid(True, which='both', ls=':', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Display seed photon fields
print("Seed Photon Fields for IC Scattering:")
print("=" * 50)
for name, T, w in SEED_PHOTON_FIELDS:
    print(f"  {name}: T = {T}, w = {w}")

## 2. Set Up MCMC Fitting

Parameters to fit:
1. `log10(N0)` - log10 of normalization at 1 TeV [1/eV]
2. `Γ` - electron spectral index
3. `log10(E_cut/TeV)` - log10 of cutoff energy

In [ ]:
# Initial parameters (near expected values from paper)
p0 = get_initial_params_ecpl()
print(f"Initial parameters: {p0}")

# Parameter labels for plots
labels = get_labels_ecpl()
print(f"Labels: {labels}")

# MCMC settings
NWALKERS = 32
NBURN = 100    # Burn-in steps (increase for production)
NRUN = 500     # Production steps (increase for production)

In [ ]:
# Run the MCMC sampler
sampler, pos = naima.run_sampler(
    data_table=data,
    p0=p0,
    labels=labels,
    model=pwn_ic_ecpl,
    prior=pwn_lnprior_ecpl,
    nwalkers=NWALKERS,
    nburn=NBURN,
    nrun=NRUN,
    threads=4,
    prefit=True,
)

## 3. Analyze Results

In [ ]:
# Save diagnostic plots
output_prefix = "pwn_ic_ecpl"
naima.save_diagnostic_plots(output_prefix, sampler, sed=True)
naima.save_results_table(output_prefix, sampler)
print(f"Saved diagnostic plots and results to {output_prefix}_*")

In [ ]:
# Extract best-fit and uncertainties
chain = sampler.get_chain(flat=True)
logp = sampler.get_log_prob(flat=True)

# Maximum a posteriori (MAP) estimate
best_idx = np.nanargmax(logp)
pars_map = chain[best_idx]

# Percentile estimates
q16, q50, q84 = np.percentile(chain, [16, 50, 84], axis=0)

print("=" * 60)
print("Best-fit Parameters (ECPL Inverse Compton Model)")
print("=" * 60)
for i, label in enumerate(labels):
    print(f"{label}: {q50[i]:.3f} (+{q84[i]-q50[i]:.3f} / -{q50[i]-q16[i]:.3f})")
print()
print(f"E_cut = {10**q50[2]:.1f} (+{10**q84[2]-10**q50[2]:.1f} / -{10**q50[2]-10**q16[2]:.1f}) TeV")

In [ ]:
# Compute total electron energy from posterior
n_samples = min(500, len(chain))
idx = np.random.choice(len(chain), size=n_samples, replace=False)

We_samples = []
for i in idx:
    try:
        We = compute_electron_energy(chain[i], model='ecpl', Emin=1*u.GeV)
        We_samples.append(We.to(u.erg).value)
    except:
        pass

We_samples = np.array(We_samples)
We_q16, We_q50, We_q84 = np.percentile(We_samples, [16, 50, 84])

print(f"\nTotal electron energy W_e (E > 1 GeV):")
print(f"  W_e = {We_q50:.2e} (+{We_q84-We_q50:.2e} / -{We_q50-We_q16:.2e}) erg")
print(f"\nCompare to Hnatyk et al. 2022: W_e = 8.80e49 erg (alternative ECPL model)")

In [ ]:
# Compute chi-squared
model_flux = pwn_ic_ecpl(pars_map, data)
residuals = (model_flux - data['flux']) / data['flux_error']
chi2 = np.sum(residuals.value**2)
ndf = len(data) - len(pars_map)

print(f"\nGoodness of fit:")
print(f"  χ² = {chi2:.2f}")
print(f"  ndf = {ndf}")
print(f"  χ²/ndf = {chi2/ndf:.2f}")

## 4. Plot Best-Fit SED with IC Components

In [ ]:
# Create energy grid for model
E_plot = np.logspace(-1, 5, 200) * u.GeV
grid = QTable({'energy': E_plot})

# Calculate total IC model at best-fit
model_flux_plot = pwn_ic_ecpl(pars_map, grid)
model_sed = (E_plot**2 * model_flux_plot).to(u.Unit('GeV/(cm2 s)'))

# Calculate individual IC components
log10_norm, index, log10_ecut = pars_map
amplitude = (10.0 ** log10_norm) / u.eV
E0 = 1.0 * u.TeV
Ecut = (10.0 ** log10_ecut) * u.TeV
electron_dist = ExponentialCutoffPowerLaw(amplitude, E0, index, Ecut)

# IC for each seed field separately
ic_cmb = InverseCompton(electron_dist, seed_photon_fields=[SEED_PHOTON_FIELDS[0]])
ic_fir = InverseCompton(electron_dist, seed_photon_fields=[SEED_PHOTON_FIELDS[1]])
ic_nir = InverseCompton(electron_dist, seed_photon_fields=[SEED_PHOTON_FIELDS[2]])

flux_cmb = ic_cmb.flux(grid, distance=PWN_DEFAULTS['distance'])
flux_fir = ic_fir.flux(grid, distance=PWN_DEFAULTS['distance'])
flux_nir = ic_nir.flux(grid, distance=PWN_DEFAULTS['distance'])

sed_cmb = (E_plot**2 * flux_cmb).to(u.Unit('GeV/(cm2 s)'))
sed_fir = (E_plot**2 * flux_fir).to(u.Unit('GeV/(cm2 s)'))
sed_nir = (E_plot**2 * flux_nir).to(u.Unit('GeV/(cm2 s)'))

# Data SED
data_E = data['energy'].to(u.GeV)
data_sed = (data_E**2 * data['flux']).to(u.Unit('GeV/(cm2 s)'))
data_sed_err = (data_E**2 * data['flux_error']).to(u.Unit('GeV/(cm2 s)'))

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(10, 7))

# Data
ax.errorbar(data_E.value, data_sed.value, yerr=data_sed_err.value,
            fmt='o', color='red', ecolor='red', capsize=3,
            label='Fermi-LAT + H.E.S.S. data', zorder=10)

# Total IC
ax.plot(E_plot.value, model_sed.value, 'b-', lw=2.5,
        label=f'Total IC (Γ={q50[1]:.2f}, E_cut={10**q50[2]:.0f} TeV)')

# Individual components
ax.plot(E_plot.value, sed_cmb.value, 'g--', lw=1.5, alpha=0.7,
        label='IC (CMB)')
ax.plot(E_plot.value, sed_fir.value, 'm--', lw=1.5, alpha=0.7,
        label='IC (FIR)')
ax.plot(E_plot.value, sed_nir.value, 'c--', lw=1.5, alpha=0.7,
        label='IC (NIR/Starlight)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(0.1, 1e5)
ax.set_ylim(1e-13, 1e-8)
ax.set_xlabel('Photon Energy [GeV]', fontsize=12)
ax.set_ylabel(r'$E^2 dN/dE$ [GeV cm$^{-2}$ s$^{-1}$]', fontsize=12)
ax.set_title('PWN Leptonic Model: Inverse Compton (ECPL Electron Spectrum)', fontsize=14)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, which='both', ls=':', alpha=0.5)

plt.tight_layout()
plt.savefig('pwn_ic_ecpl_sed.png', dpi=150)
plt.show()

print("Saved SED plot to pwn_ic_ecpl_sed.png")

## 5. Summary Table (like Table 1 in paper)

In [ ]:
# Build summary table
summary_rows = [
    ('E_e,min [GeV] (fixed)', 1.0, '-', '-'),
    ('E_e,max [GeV] (fixed)', 1e6, '-', '-'),
    ('Distance [kpc] (fixed)', PWN_DEFAULTS['distance'].value, '-', '-'),
    ('log₁₀(N₀/eV⁻¹)', f'{q50[0]:.2f}', f'+{q84[0]-q50[0]:.2f}', f'-{q50[0]-q16[0]:.2f}'),
    ('Γ_e', f'{q50[1]:.2f}', f'+{q84[1]-q50[1]:.2f}', f'-{q50[1]-q16[1]:.2f}'),
    ('E_cut [TeV]', f'{10**q50[2]:.1f}', f'+{10**q84[2]-10**q50[2]:.1f}', f'-{10**q50[2]-10**q16[2]:.1f}'),
    ('W_e (>1 GeV) [erg]', f'{We_q50:.2e}', f'+{We_q84-We_q50:.2e}', f'-{We_q50-We_q16:.2e}'),
    ('χ²/ndf', f'{chi2/ndf:.2f}', '-', '-'),
]

print("\n" + "=" * 70)
print("Summary: PWN ECPL Inverse Compton Model (1 Population)")
print("=" * 70)
print(f"{'Parameter':<25} {'Value':>15} {'Upper':>12} {'Lower':>12}")
print("-" * 70)
for row in summary_rows:
    print(f"{row[0]:<25} {row[1]:>15} {row[2]:>12} {row[3]:>12}")
print("=" * 70)

print("\nSeed Photon Fields:")
for name, T, w in SEED_PHOTON_FIELDS:
    print(f"  {name}: T = {T}, w = {w}")

## 6. Comparison with Paper Results

From Hnatyk et al. 2022, Table 1 (Alternative ECPL model for PWN):
- Γ = 3.08 ± 0.03
- E_cut = 424.3 ± 21.1 TeV  
- W_e = 8.80×10⁴⁹ erg
- χ²/ndf = 40.24/36 = 1.12

In [ ]:
print("\nComparison with Hnatyk et al. 2022 (Alternative ECPL):")
print("=" * 50)
print(f"{'Parameter':<15} {'This work':>15} {'Paper':>15}")
print("-" * 50)
print(f"{'Γ_e':<15} {q50[1]:>15.2f} {3.08:>15.2f}")
print(f"{'E_cut [TeV]':<15} {10**q50[2]:>15.1f} {424.3:>15.1f}")
print(f"{'W_e [erg]':<15} {We_q50:>15.2e} {8.80e49:>15.2e}")
print(f"{'χ²/ndf':<15} {chi2/ndf:>15.2f} {1.12:>15.2f}")
print("=" * 50)